# 00 — Data Fetching

Fetches the pedestrian street network and raw OSM features for five inner London boroughs.  
Exports one CSV per borough: `data/raw/raw_{borough}.csv`

**Features collected per street segment:**
| Column | Source | Notes |
|---|---|---|
| `lamp_count` | Mapillary CSV (user-supplied) | Streetlights near segment |
| `node_degree_u` / `node_degree_v` | OSMnx graph | Connectivity at each endpoint |
| `building_coverage` | OSM buildings | Footprint area fraction in 50 m buffer |
| `street_width_m` | OSM `width` tag | Falls back to NaN if untagged |
| `commercial_count` | OSM amenities + landuse | Commercial/retail/food amenities in 100 m |
| `residential_flag` | OSM landuse | 1 if residential landuse overlaps 50 m buffer |
| `transit_dist_m` | OSM transit stops | Distance (m) to nearest stop |

> **Lighting note:** street lamp data comes from a pre-existing Mapillary CSV.  
> Set `MAPILLARY_CSV` below to its path before running.

In [2]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

ox.settings.log_console = False
ox.settings.use_cache = True

In [3]:
# ── Paths ────────────────────────────────────────────────────────────────────
REPO_ROOT     = Path('..').resolve()
RAW_DIR       = REPO_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Path to the Mapillary streetlight CSV.
# Expected columns: latitude, longitude (any extra columns are ignored).
MAPILLARY_CSV = Path(r'C:\Users\maria\Documents\GitHub\MaCAD26-G01-DataEncoding\csv\mapillary_streetlights_london_ids.csv').resolve()

# ── Borough definitions ───────────────────────────────────────────────────────
BOROUGHS = {
    'westminster':    'City of Westminster, London, UK',
    'islington':      'London Borough of Islington, London, UK',
    'hackney':        'London Borough of Hackney, London, UK',
    'tower_hamlets':  'London Borough of Tower Hamlets, London, UK',
    'southwark':      'London Borough of Southwark, London, UK',
}

# ── Spatial parameters ────────────────────────────────────────────────────────
CRS_PROJECTED  = 'EPSG:27700'  # British National Grid — metres
BUFFER_SMALL   = 50            # metres  — buildings, lamps, residential
BUFFER_LARGE   = 100           # metres  — commercial amenities
BUFFER_TRANSIT = 400           # metres  — max transit search radius

print('Config ready.')

Config ready.


In [4]:
# ── Helper: edges GeoDataFrame from OSMnx graph ───────────────────────────────
def graph_to_edges_gdf(G):
    """Return projected edge GeoDataFrame with basic attributes."""
    nodes, edges = ox.graph_to_gdfs(G)
    edges = edges[['geometry', 'length', 'name', 'highway', 'width']].copy()
    edges['street_width_m'] = pd.to_numeric(edges['width'], errors='coerce')
    edges = edges.drop(columns='width')
    # Add node degrees
    degree = dict(G.degree())
    edges['node_degree_u'] = edges.index.get_level_values('u').map(degree)
    edges['node_degree_v'] = edges.index.get_level_values('v').map(degree)
    return edges.to_crs(CRS_PROJECTED)

In [5]:
# ── Helper: load and project Mapillary lights ─────────────────────────────────
def load_mapillary_lights():
    """Load Mapillary CSV and return a projected GeoDataFrame of light points."""
    if not MAPILLARY_CSV.exists():
        print(f'  [WARNING] Mapillary CSV not found at {MAPILLARY_CSV}.')
        print('  lamp_count will be set to NaN for all segments.')
        return None
    df = pd.read_csv(MAPILLARY_CSV)
    # Normalise column names to lowercase
    df.columns = df.columns.str.lower().str.strip()
    # Accept 'lat'/'lon' or 'latitude'/'longitude'
    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    if lat_col is None or lon_col is None:
        raise ValueError(f'Could not find lat/lon columns in {MAPILLARY_CSV}. Found: {list(df.columns)}')
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    ).to_crs(CRS_PROJECTED)
    print(f'  Loaded {len(gdf):,} Mapillary light points.')
    return gdf

In [6]:
# ── Helper: count points within buffer of each segment ────────────────────────
def count_points_in_buffer(edges_proj, points_gdf, buffer_m, col_name):
    """Add a column counting how many points fall within buffer_m of each edge."""
    if points_gdf is None:
        edges_proj[col_name] = np.nan
        return edges_proj
    buffers = edges_proj.geometry.buffer(buffer_m)
    counts = []
    for buf in buffers:
        counts.append(points_gdf[points_gdf.geometry.within(buf)].shape[0])
    edges_proj[col_name] = counts
    return edges_proj

In [7]:
# ── Helper: building coverage fraction ───────────────────────────────────────
def add_building_coverage(edges_proj, borough_query):
    """Fraction of 50 m buffer area covered by building footprints."""
    print('  Fetching buildings...')
    try:
        bldgs = ox.features_from_place(borough_query, tags={'building': True})
        bldgs = bldgs[bldgs.geometry.type.isin(['Polygon', 'MultiPolygon'])]
        bldgs = bldgs[['geometry']].to_crs(CRS_PROJECTED)
    except Exception as e:
        print(f'  [WARNING] Could not fetch buildings: {e}')
        edges_proj['building_coverage'] = np.nan
        return edges_proj

    coverage = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_SMALL)
        buf_area = buf.area
        clipped = bldgs[bldgs.geometry.intersects(buf)].geometry.intersection(buf)
        cov = clipped.area.sum() / buf_area if buf_area > 0 else 0.0
        coverage.append(round(cov, 4))
    edges_proj['building_coverage'] = coverage
    return edges_proj

In [8]:
# ── Helper: commercial amenity count ─────────────────────────────────────────
COMMERCIAL_AMENITY_TAGS = {
    'amenity': [
        'restaurant', 'cafe', 'bar', 'pub', 'fast_food', 'food_court',
        'marketplace', 'bank', 'pharmacy', 'cinema', 'theatre',
        'nightclub', 'casino', 'gym',
    ],
    'shop': True,
    'office': True,
}

def add_commercial_count(edges_proj, borough_query):
    """Count commercial amenities within BUFFER_LARGE of each segment."""
    print('  Fetching commercial amenities...')
    try:
        amenities = ox.features_from_place(borough_query, tags=COMMERCIAL_AMENITY_TAGS)
        amenities = amenities[['geometry']].copy()
        # Represent polygons as centroids for simplicity
        amenities['geometry'] = amenities.geometry.apply(
            lambda g: g.centroid if g.geom_type != 'Point' else g
        )
        amenities = amenities.to_crs(CRS_PROJECTED)
    except Exception as e:
        print(f'  [WARNING] Could not fetch amenities: {e}')
        edges_proj['commercial_count'] = np.nan
        return edges_proj

    edges_proj = count_points_in_buffer(edges_proj, amenities, BUFFER_LARGE, 'commercial_count')
    return edges_proj

In [9]:
# ── Helper: residential flag ──────────────────────────────────────────────────
def add_residential_flag(edges_proj, borough_query):
    """1 if a residential landuse polygon overlaps the 50 m buffer, else 0."""
    print('  Fetching residential landuse...')
    try:
        residential = ox.features_from_place(
            borough_query,
            tags={'landuse': ['residential', 'apartments', 'housing']}
        )
        residential = residential[
            residential.geometry.type.isin(['Polygon', 'MultiPolygon'])
        ][['geometry']].to_crs(CRS_PROJECTED)
    except Exception as e:
        print(f'  [WARNING] Could not fetch residential landuse: {e}')
        edges_proj['residential_flag'] = np.nan
        return edges_proj

    flags = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_SMALL)
        flag = int(residential[residential.geometry.intersects(buf)].shape[0] > 0)
        flags.append(flag)
    edges_proj['residential_flag'] = flags
    return edges_proj

In [10]:
# ── Helper: transit proximity ─────────────────────────────────────────────────
# London transit: bus stops, Tube/Overground/Elizabeth line stations, tram stops
TRANSIT_TAGS = {
    'highway': 'bus_stop',
    'railway': ['station', 'subway_entrance', 'tram_stop', 'halt'],
    'public_transport': ['stop_position', 'platform'],
}

def add_transit_dist(edges_proj, borough_query):
    """Distance in metres to the nearest transit stop (EPSG:27700 is in metres)."""
    print('  Fetching transit stops...')
    try:
        stops = ox.features_from_place(borough_query, tags=TRANSIT_TAGS)
        stops['geometry'] = stops.geometry.apply(
            lambda g: g.centroid if g.geom_type != 'Point' else g
        )
        stops = stops[['geometry']].to_crs(CRS_PROJECTED)
        stops = stops.drop_duplicates(subset='geometry')
    except Exception as e:
        print(f'  [WARNING] Could not fetch transit stops: {e}')
        edges_proj['transit_dist_m'] = np.nan
        return edges_proj

    # For each segment midpoint, find nearest stop
    midpoints = edges_proj.geometry.interpolate(0.5, normalized=True)
    stop_geoms = np.array([(g.x, g.y) for g in stops.geometry])
    dists = []
    for mp in midpoints:
        if len(stop_geoms) == 0:
            dists.append(np.nan)
        else:
            d = np.sqrt(((stop_geoms - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
            dists.append(round(float(d), 2))  # already metres in EPSG:27700
    edges_proj['transit_dist_m'] = dists
    return edges_proj

In [ ]:
# ── Main fetch loop ───────────────────────────────────────────────────────────
lights_gdf = load_mapillary_lights()

for borough_key, borough_query in BOROUGHS.items():
    out_path = RAW_DIR / f'raw_{borough_key}.csv'
    if out_path.exists():
        print(f'\n[SKIP] {borough_key} — already exists at {out_path}')
        continue

    print(f'\n══ {borough_key.upper()} ══')

    # 1. Street network
    print('  Fetching walk network...')
    G = ox.graph_from_place(borough_query, network_type='walk', simplify=True)
    G = ox.distance.add_edge_lengths(G)
    edges = graph_to_edges_gdf(G)
    edges['borough'] = borough_key
    print(f'  {len(edges):,} edges loaded.')

    # 2. Lighting (Mapillary)
    print('  Attaching Mapillary light counts...')
    if lights_gdf is not None:
        bbox = edges.total_bounds  # minx, miny, maxx, maxy in projected CRS
        local_lights = lights_gdf.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
        print(f'  {len(local_lights):,} lights within borough bbox.')
    else:
        local_lights = None
    edges = count_points_in_buffer(edges, local_lights, BUFFER_SMALL, 'lamp_count')

    # 3. Building coverage
    edges = add_building_coverage(edges, borough_query)

    # 4. Commercial amenities
    edges = add_commercial_count(edges, borough_query)

    # 5. Residential flag
    edges = add_residential_flag(edges, borough_query)

    # 6. Transit proximity
    edges = add_transit_dist(edges, borough_query)

    # ── Export ────────────────────────────────────────────────────────────────
    for col in ['name', 'highway']:
        if col in edges.columns:
            edges[col] = edges[col].apply(
                lambda x: '|'.join(x) if isinstance(x, list) else x
            )

    edges['geometry_wkt'] = edges.geometry.to_wkt()
    df_out = pd.DataFrame(edges.drop(columns='geometry'))
    df_out.to_csv(out_path, index=True)  # index = (u, v, key)
    print(f'  Saved → {out_path.name}  ({len(df_out):,} rows)')

  Loaded 61,784 Mapillary light points.

══ WESTMINSTER ══
  Fetching walk network...


AttributeError: module 'osmnx' has no attribute 'add_edge_lengths'

In [ ]:
# ── Validation summary ────────────────────────────────────────────────────────
print('\n── Raw data summary ──')
total = 0
for borough_key in BOROUGHS:
    p = RAW_DIR / f'raw_{borough_key}.csv'
    if p.exists():
        df = pd.read_csv(p, index_col=[0, 1, 2])
        total += len(df)
        nan_pct = df.isna().mean().mul(100).round(1)
        print(f'\n{borough_key}: {len(df):,} segments')
        print(nan_pct[nan_pct > 0].to_string() or '  No missing values.')
    else:
        print(f'{borough_key}: NOT FOUND')
print(f'\nTotal segments across all boroughs: {total:,}')